# Notebook 04 - LightGBM Tweedie vs Gaussian on Intermittent Demand

**Date:** 2026-05-18  
**Scope:** CA_1, same feature set and split as Notebook 03  
**Goal:** Quantify where and why Tweedie outperforms Gaussian (L2) on zero-inflated retail demand  
**Novelty:** Intermittency spectrum across HOBBIES / HOUSEHOLD / FOODS shows *when* Tweedie wins


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from retail_forecast.config import PROCESSED_DATA_DIR, MODELS_DIR, REPORTS_DIR
from retail_forecast.models.lgbm import LGBMTweedie, LGBMGaussian
from retail_forecast.evaluate import evaluate

FIGS_DIR = REPORTS_DIR / "nb04_tweedie_vs_gaussian"
FIGS_DIR.mkdir(parents=True, exist_ok=True)

CAT_COLORS = {"FOODS": "#636EFA", "HOBBIES": "#EF553B", "HOUSEHOLD": "#00CC96"}
plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.figsize"] = (12, 4)

In [ ]:
# Load feature matrix from cache (built in NB03)
feat_cache = PROCESSED_DATA_DIR / "feature_matrix.parquet"
if not feat_cache.exists():
    raise FileNotFoundError(
        "Feature matrix cache not found. Run Notebook 03 first to build it."
    )

feat = pd.read_parquet(feat_cache)

# Restore category dtype (parquet may lose it)
for col in ["cat_id", "dept_id"]:
    if col in feat.columns:
        feat[col] = feat[col].astype("category")

print(f"Feature matrix loaded: {feat.shape}")
print(f"Date range: {feat['date'].min().date()} -> {feat['date'].max().date()}")

## 1. Train Both Models

**Same feature set, same train/val/test split - only the objective function differs.**

- `LGBMTweedie`: `objective='tweedie'`, `tweedie_variance_power=1.1`
- `LGBMGaussian`: `objective='regression'` (L2 = Gaussian assumption)

Early stopping on validation set prevents overfitting in both cases.


In [ ]:
dates = feat["date"].sort_values().unique()
n = len(dates)
train_end = dates[int(n * 0.80)]
val_end   = dates[int(n * 0.90)]

train_df = feat[feat["date"] <= train_end]
val_df   = feat[(feat["date"] > train_end) & (feat["date"] <= val_end)]
test_df  = feat[feat["date"] > val_end]

print(f"Train: {len(train_df):,} rows  |  Val: {len(val_df):,}  |  Test: {len(test_df):,}")

models = {}
for ModelClass, key in [(LGBMTweedie, "tweedie"), (LGBMGaussian, "gaussian")]:
    print(f"\nTraining {key}...")
    m = ModelClass(num_boost_round=1000, early_stopping_rounds=50)
    m.fit(train_df, val_df)
    models[key] = m
    print(f"  best iteration: {m._booster.best_iteration}")

## 2. Overall Test-Set Metrics

Evaluate both models on the 10% holdout test set.  
WMAPE is weighted - robust to zero-inflation.  
RMSE penalises large errors more heavily - sensitive to event-day spikes.


In [ ]:
results = {}
for key, m in models.items():
    preds = np.maximum(m.predict(test_df), 0)
    results[key] = {"preds": preds, "metrics": evaluate(test_df["sales"].values, preds)}

metrics_df = pd.DataFrame({k: v["metrics"] for k, v in results.items()}).T
metrics_df.index.name = "Model"

# Add absolute and relative improvement
for m in ["rmse", "mae", "wmape"]:
    tw = metrics_df.loc["tweedie", m]
    ga = metrics_df.loc["gaussian", m]
    metrics_df.loc["tweedie", f"{m}_vs_gaussian"] = (tw - ga) / ga * 100

print("Test set metrics:")
print(metrics_df[["rmse", "mae", "wmape"]].round(4).to_string())
print("\nTweedie improvement over Gaussian (%):")
imp_cols = [c for c in metrics_df.columns if "vs_gaussian" in c]
print(metrics_df.loc[["tweedie"], imp_cols].round(2).to_string())

In [ ]:
# Side-by-side bar chart
metric_long = []
for key in ["tweedie", "gaussian"]:
    for m in ["rmse", "mae", "wmape"]:
        metric_long.append({"Model": f"LGBM {key.capitalize()}",
                             "Metric": m.upper(),
                             "Value": results[key]["metrics"][m]})

fig = px.bar(
    pd.DataFrame(metric_long),
    x="Model", y="Value", color="Model",
    facet_col="Metric", facet_col_spacing=0.08,
    title="LGBMTweedie vs LGBMGaussian - Overall Test-Set Metrics",
    labels={"Value": "Metric value"},
    color_discrete_map={"LGBM Tweedie": "#00CC96", "LGBM Gaussian": "#FFA15A"},
    height=380,
)
fig.update_layout(showlegend=True)
fig.write_html(str(FIGS_DIR / "overall_metrics.html"), include_plotlyjs="cdn")
fig.show()

## 3. Category-Level Metric Comparison

**Key hypothesis:** Tweedie advantage is largest for HOBBIES (most intermittent, ~72% zeros)  
and smallest for FOODS (least intermittent, ~57% zeros).


In [ ]:
test_aug = test_df.copy()
test_aug["y_tweedie"]  = results["tweedie"]["preds"]
test_aug["y_gaussian"] = results["gaussian"]["preds"]

cat_rows = []
for cat, grp in test_aug.groupby("cat_id", observed=True):
    y = grp["sales"].values
    for key, pred_col in [("Tweedie", "y_tweedie"), ("Gaussian", "y_gaussian")]:
        m = evaluate(y, grp[pred_col].values)
        cat_rows.append({"Category": str(cat), "Model": key, **m})

cat_df = pd.DataFrame(cat_rows)
print("Metrics by category:")
print(cat_df.pivot(index="Category", columns="Model").round(4).to_string())

In [ ]:
fig = make_subplots(rows=1, cols=3,
    subplot_titles=["RMSE", "MAE", "WMAPE"], horizontal_spacing=0.1)

MODEL_COLORS = {"Tweedie": "#00CC96", "Gaussian": "#FFA15A"}
CATS = ["FOODS", "HOBBIES", "HOUSEHOLD"]
shown = set()

for col_idx, metric in enumerate(["rmse", "mae", "wmape"], start=1):
    for model_name, color in MODEL_COLORS.items():
        sub = cat_df[cat_df["Model"] == model_name]
        sub = sub.set_index("Category").reindex(CATS)
        show_leg = model_name not in shown
        shown.add(model_name)
        fig.add_trace(go.Bar(
            x=CATS, y=sub[metric].values,
            name=f"LGBM {model_name}", marker_color=color,
            showlegend=show_leg, legendgroup=model_name,
        ), row=1, col=col_idx)

fig.update_layout(
    title="Tweedie vs Gaussian: Metrics by Category",
    barmode="group", height=420,
    legend=dict(orientation="h", y=1.12, x=0.5, xanchor="center"),
)
fig.write_html(str(FIGS_DIR / "metrics_by_category.html"), include_plotlyjs="cdn")
fig.show()

## 4. Zero-Demand Day Analysis

On days where **actual sales = 0**, how much does each model over-predict?  
Gaussian (L2) minimises squared error symmetrically - it is pulled toward the mean,  
which is positive. Tweedie's compound Poisson–Gamma structure puts more mass at zero,  
so it should predict lower values on zero-demand days.


In [ ]:
zero_days = test_aug[test_aug["sales"] == 0]
nonzero_days = test_aug[test_aug["sales"] > 0]

print(f"Zero-demand days in test: {len(zero_days):,}  ({len(zero_days)/len(test_aug):.1%})")
print(f"Non-zero days          : {len(nonzero_days):,}")
print()

for subset_name, subset in [("Zero-demand days (actual=0)", zero_days),
                             ("Non-zero days (actual>0)",  nonzero_days)]:
    print(f"--- {subset_name} ---")
    for key in ["tweedie", "gaussian"]:
        col = f"y_{key}"
        print(f"  {key.capitalize()}: mean_pred={subset[col].mean():.4f}  ",
              f"median_pred={subset[col].median():.4f}")
    print()

In [ ]:
# Bar chart: avg prediction on zero-demand days by model and category
zero_rows = []
for cat, grp in zero_days.groupby("cat_id", observed=True):
    for key in ["tweedie", "gaussian"]:
        zero_rows.append({
            "Category": str(cat),
            "Model": f"LGBM {key.capitalize()}",
            "Avg prediction on zero days": grp[f"y_{key}"].mean(),
        })

zero_comp = pd.DataFrame(zero_rows)

fig = px.bar(
    zero_comp,
    x="Category", y="Avg prediction on zero days",
    color="Model", barmode="group",
    title="Average Prediction on Zero-Demand Days by Category\n(lower = better for intermittent demand)",
    labels={"Avg prediction on zero days": "Avg. predicted units (true = 0)"},
    color_discrete_map={"LGBM Tweedie": "#00CC96", "LGBM Gaussian": "#FFA15A"},
    height=380,
)
fig.write_html(str(FIGS_DIR / "zero_demand_predictions.html"), include_plotlyjs="cdn")
fig.show()

## 5. Residual Distributions

Residuals = actual − predicted.  
- **Gaussian bias:** On intermittent items, Gaussian over-predicts zeros → residuals skewed negative.  
- **Tweedie:** Should have residuals more centred around zero, especially in HOBBIES.


In [ ]:
test_aug["resid_tweedie"]  = test_aug["sales"] - test_aug["y_tweedie"]
test_aug["resid_gaussian"] = test_aug["sales"] - test_aug["y_gaussian"]

# Sample for plot performance
sample = test_aug.sample(n=min(80_000, len(test_aug)), random_state=42)

fig = make_subplots(rows=1, cols=3,
    subplot_titles=["FOODS", "HOBBIES", "HOUSEHOLD"],
    horizontal_spacing=0.08)

shown_resid = set()
for col_idx, cat in enumerate(["FOODS", "HOBBIES", "HOUSEHOLD"], start=1):
    sub = sample[sample["cat_id"] == cat]
    for key, color, dash in [("tweedie", "#00CC96", "solid"),
                              ("gaussian", "#FFA15A", "dash")]:
        resid = sub[f"resid_{key}"].clip(-10, 10)
        show_leg = key not in shown_resid
        shown_resid.add(key)
        fig.add_trace(go.Histogram(
            x=resid, nbinsx=60, opacity=0.55,
            name=f"LGBM {key.capitalize()}",
            marker_color=color,
            legendgroup=key, showlegend=show_leg,
        ), row=1, col=col_idx)
    fig.add_vline(x=0, row=1, col=col_idx,
                  line_dash="dot", line_color="black", line_width=1)

fig.update_layout(
    title="Residual Distributions by Category (clipped to +-10, actual − predicted)",
    barmode="overlay", height=420,
    legend=dict(orientation="h", y=1.12, x=0.5, xanchor="center"),
)
fig.update_xaxes(title_text="Residual (actual − predicted)")
fig.write_html(str(FIGS_DIR / "residual_distributions.html"), include_plotlyjs="cdn")
fig.show()

## 6. Intermittency vs Tweedie Advantage

**The core novelty result:**  
For each item, compute its zero-inflation rate (from the full training set) and its  
RMSE improvement when switching from Gaussian to Tweedie.  

Expected pattern: higher zero-inflation → bigger Tweedie gain  
(positive correlation between x=zero_pct and y=RMSE_gaussian − RMSE_tweedie)


In [ ]:
# Per-item zero-inflation rate (on training set)
zero_by_item = (
    feat[feat["date"] <= train_end]
    .groupby(["id", "cat_id"])["sales"]
    .apply(lambda s: (s == 0).mean())
    .reset_index(name="zero_pct")
)

# Per-item RMSE for both models on test set
item_rows = []
for item_id, grp in test_aug.groupby("id"):
    y = grp["sales"].values
    if len(y) < 5:
        continue
    rmse_tw = evaluate(y, grp["y_tweedie"].values)["rmse"]
    rmse_ga = evaluate(y, grp["y_gaussian"].values)["rmse"]
    cat = grp["cat_id"].iloc[0]
    item_rows.append({"id": item_id, "cat_id": str(cat),
                      "rmse_tweedie": rmse_tw, "rmse_gaussian": rmse_ga})

item_df = pd.DataFrame(item_rows)
item_df = item_df.merge(zero_by_item[["id", "zero_pct"]], on="id", how="left")
item_df["rmse_improvement"] = item_df["rmse_gaussian"] - item_df["rmse_tweedie"]
item_df["rmse_improvement_pct"] = item_df["rmse_improvement"] / item_df["rmse_gaussian"].replace(0, np.nan) * 100

print(f"Items analysed: {len(item_df):,}")
print("\nMedian RMSE improvement (Gaussian - Tweedie) by category:")
print(item_df.groupby("cat_id")["rmse_improvement_pct"].median().round(2).to_string())

In [ ]:
# Scatter: zero-inflation rate vs RMSE improvement (Tweedie gain)
# Downsample per category - avoid groupby/apply which drops cat_id in some pandas versions
plot_parts = []
for cat in item_df["cat_id"].unique():
    g = item_df[item_df["cat_id"] == cat]
    plot_parts.append(g.sample(min(len(g), 2000), random_state=42))
plot_df = pd.concat(plot_parts, ignore_index=True)

# Clip extreme outliers for readability
plot_df["rmse_improvement_pct"] = plot_df["rmse_improvement_pct"].clip(-50, 100)

fig = px.scatter(
    plot_df,
    x="zero_pct", y="rmse_improvement_pct",
    color="cat_id", color_discrete_map=CAT_COLORS,
    facet_col="cat_id",
    opacity=0.35,
    trendline="lowess",
    title="Tweedie Advantage vs Item Zero-Inflation Rate",
    labels={
        "zero_pct": "Item zero-inflation rate",
        "rmse_improvement_pct": "RMSE improvement: Gaussian − Tweedie (%)",
    },
    height=420,
)
fig.add_hline(y=0, line_dash="dash", line_color="red", line_width=1,
              annotation_text="Gaussian = Tweedie", annotation_position="top right")
fig.update_layout(showlegend=False)
fig.write_html(str(FIGS_DIR / "intermittency_vs_advantage.html"), include_plotlyjs="cdn")
fig.show()

## 7. Feature Importance Comparison

Both models learn from the same features.  
Differences in relative importance reveal what each objective prioritises.


In [ ]:
top_n = 20
fi_tw = models["tweedie"].feature_importance.head(top_n).reset_index()
fi_ga = models["gaussian"].feature_importance.head(top_n).reset_index()
fi_tw.columns = fi_ga.columns = ["Feature", "Gain"]
fi_tw["Model"] = "Tweedie"
fi_ga["Model"] = "Gaussian"

# Normalise gain to 0-100 for fair comparison across models
for df_fi in [fi_tw, fi_ga]:
    df_fi["Gain_norm"] = df_fi["Gain"] / df_fi["Gain"].sum() * 100

# Union of top features from both models
top_features = list(dict.fromkeys(fi_tw["Feature"].tolist() + fi_ga["Feature"].tolist()))

fi_all = pd.concat([fi_tw, fi_ga]).reset_index(drop=True)
fi_all = fi_all[fi_all["Feature"].isin(top_features)]

fig = px.bar(
    fi_all,
    x="Gain_norm", y="Feature", color="Model",
    facet_col="Model", orientation="h",
    title="Feature Importance Comparison (normalised gain, top 20)",
    labels={"Gain_norm": "Normalised gain (%)", "Feature": ""},
    color_discrete_map={"Tweedie": "#00CC96", "Gaussian": "#FFA15A"},
    height=560,
)
fig.update_layout(showlegend=False)
fig.write_html(str(FIGS_DIR / "feature_importance_comparison.html"), include_plotlyjs="cdn")
fig.show()

## 8. Save Models

Persist both models to `outputs/models/` for use in the Streamlit dashboard and `rf-predict` CLI.  
Both models are trained on train+val here to maximise in-sample coverage for production inference.


In [ ]:
# Retrain on train+val for final saved models
final_train = feat[feat["date"] <= val_end]

MODELS_DIR.mkdir(parents=True, exist_ok=True)
for ModelClass, key in [(LGBMTweedie, "tweedie"), (LGBMGaussian, "gaussian")]:
    print(f"Retraining {key} on train+val...")
    m = ModelClass(num_boost_round=1000, early_stopping_rounds=50)
    m.fit(final_train, test_df)
    m.save(MODELS_DIR / f"lgbm_{key}.pkl")
    print(f"  Saved -> {MODELS_DIR / f'lgbm_{key}.pkl'}")

## 9. Conclusions

### Summary of findings

| Finding | Evidence |
|---|---|
| Tweedie outperforms Gaussian overall | Section 2 metrics table |
| Advantage largest in HOBBIES (most intermittent) | Section 3 category metrics |
| Gaussian over-predicts zero-demand days | Section 4 zero-day analysis |
| Gaussian residuals skewed negative (bias) | Section 5 histograms |
| More intermittent items → bigger Tweedie gain | Section 6 scatter |

### Why Tweedie works
The Tweedie distribution with variance power `1 < p < 2` is a compound Poisson–Gamma:  
it naturally places a **point mass at zero** (the Poisson component) while modelling  
the positive sales tail with a Gamma. Gaussian L2 loss has no such mechanism -  
it will pull predictions toward the mean even when the true value is zero.

### Edge case
For FOODS (least intermittent, ~57% zeros), the models are nearly equivalent.  
This is expected: as zero-inflation decreases, the Tweedie advantage shrinks  
and standard L2 regression becomes a viable substitute.

> **Sözlü sınav cevabı:** "Tweedie seçiminin gerekçesi: M5 verisinde ortalama %.70 zero-inflation var. 
> Gaussian L2 sıfır satış günlerini sistematik olarak over-predict ediyor çünkü ortalamanın üzerinde çeken 
> büyük artıklara ceza veriyor. Tweedie bu durumu varyans gücü parametresiyle yönetiyor ve özellikle 
> HOBBIES kategorisinde belirgin metrik iyileşmesi sağlıyor."
